# RAG — Stage 2: Indexing (embeddings + FAISS)

## Stage 2: Indexing

We split the document into chunks and store embeddings in FAISS for retrieval.

**Goal:** map each text chunk to a **dense vector** (OpenAI embeddings) and store them in a **FAISS** index for fast similarity search.

**Requirements:** `OPENAI_API_KEY` in `rag_lab/.env` (embeddings are billed).

**What you will see:** chunk count, embedding model name, in-memory FAISS, then **persist** to `data/faiss_cache/jdeco_srs/` (`index.faiss` + `index.pkl` + docstore).

**Next notebook:** Stage 3 **loads** that saved index and runs top-k retrieval (no re-embedding if the cache exists).

**About the filename:** `02_rag_stage2_indexing.executed.ipynb` is the Stage 2 lab notebook saved **after a run** (outputs stored in the file — it can look like a “second” variant, but it is the same lab). Stages 1 and 2 use this `.executed` naming; Stage 3/4 use plain `.ipynb`.

**Why the run is slow:** each chunk is embedded via the **OpenAI embeddings API** (many chunks ⇒ many API calls). Set `FORCE_REBUILD = False` in the code cell to **load** `data/faiss_cache/jdeco_srs/` when it already exists and skip re-embedding (fast). Use `True` after you change the PDF, chunk settings, or embedding model.

**Working directory:** the first code cell locates `rag_lab` via `jupyter_bootstrap.py` (repo root `QBrain/rag_lab/`), so it works whether Jupyter’s cwd is `rag_lab/`, `rag_lab/notebooks/`, `QBrain/`, or the workspace root — not only when cwd is `notebooks/`.


In [1]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import runpy
from pathlib import Path

_cwd = Path.cwd().resolve()
_roots = []
if _cwd.name == "notebooks":
    _roots.append(_cwd.parent)
_roots.extend(
    [
        _cwd,
        _cwd / "QBrain" / "rag_lab",
        _cwd / "rag_lab",
        _cwd.parent,
    ]
)
_roots.extend(_cwd.parents)
_jb = None
for _r in _roots:
    _p = Path(_r).resolve() / "jupyter_bootstrap.py"
    if _p.is_file() and (_p.parent / "src" / "qbrain_rag").is_dir():
        _jb = _p
        break
if _jb is None:
    raise RuntimeError(
        "Could not find rag_lab/jupyter_bootstrap.py. "
        f"Open QBrain/rag_lab or QBrain/rag_lab/notebooks as the working folder. cwd={_cwd!s}"
    )
_ns = runpy.run_path(str(_jb))
RAG_LAB = _ns["RAG_LAB"]

from qbrain_rag.application.chunking import chunk_text
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path
from qbrain_rag.config.settings import get_settings
from qbrain_rag.infrastructure.document_loaders import load_document
from qbrain_rag.infrastructure.vector_store import (
    build_faiss_store,
    indexed_document_count,
    load_faiss_store,
    save_faiss_store,
)

# False = reuse disk cache when present (fast). First run still builds if cache is missing.
# True = always re-embed and overwrite cache (slow; use after PDF/chunk/embed model changes).
FORCE_REBUILD = False

srs_path = resolve_default_srs_path(RAG_LAB)
text = load_document(str(srs_path))

s = get_settings()
chunk_size, chunk_overlap = s.chunk_size, s.chunk_overlap
print(f"chunk_size = {chunk_size}  chunk_overlap = {chunk_overlap}")

chunks = chunk_text(text)
print(f"Number of chunks: {len(chunks)}")
print(f"Sample chunk:\n{chunks[0]}")

metadata = [{"source_file": srs_path.name, "chunk_id": i + 1} for i in range(len(chunks))]

print("Embedding model:", s.embedding_model)

FAISS_INDEX_DIR = RAG_LAB / "data" / "faiss_cache" / "jdeco_srs"
faiss_file = FAISS_INDEX_DIR / "index.faiss"
pkl_file = FAISS_INDEX_DIR / "index.pkl"
cache_ok = faiss_file.is_file() and pkl_file.is_file()

if not FORCE_REBUILD and cache_ok:
    store = load_faiss_store(FAISS_INDEX_DIR)
    print("Loaded FAISS from disk (skipped embeddings):", FAISS_INDEX_DIR.resolve())
    print("Store type:", type(store).__name__)
    if indexed_document_count(store) != len(chunks):
        print(
            "Warning: chunk count from PDF does not match index; set FORCE_REBUILD = True to re-embed."
        )
else:
    if not FORCE_REBUILD and not cache_ok:
        print("No cache found; building index (this will call the embeddings API).")
    store = build_faiss_store(chunks, metadata)
    print("Indexed", len(chunks), "chunks into FAISS (in-memory).")
    print("Store type:", type(store).__name__)
    FAISS_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    save_faiss_store(store, FAISS_INDEX_DIR)
    print("Saved FAISS + docstore to:", FAISS_INDEX_DIR.resolve())


chunk_size = 2000  chunk_overlap = 300
Number of chunks: 30
Sample chunk:
Software Requirements 
Specification 
for 
JDECo Services Management 
system 
Version 1.0 approved 
 
Prepared by :Hana Albidaq, Shahd Muharb, Hadeel Sawalha, Salam Rabi 
 
Birzeit University 
 
February 26,2023 
 
Copyright © 2013 by Karl Wiegers and Seilevel. Permission is granted to use and modify this document. 
1 

Software Requirements Specification for JDECo Services Management system​
​
 
Table of Contents 
1. Introduction​
5 
1.1 Purpose​
5 
1.2 Document Conventions​
5 
1.3 Project Scope​
5 
1.4 References​
6 
2. Overall Description​
6 
2.1 Product Perspective​
6 
2.2 User Classes and Characteristics​
7 
2.2.1 JDECo-SMS Class Diagram​
7 
2.2.2 JDECo Use Cases​
9 
2.2.3 JEDCo-Uses Case Diagram​
9 
2.3 Operating Environment​
13 
2.4 Design and Implementation Constraints​
14 
2.5 Assumptions and Dependencies​
14 
3. System Features​
15 
3.1 JEDCo-SMS Features Scope and limitation​
16 
3.2 System Requirement

### Verify: vectors **and** chunk text

FAISS stores **embedding vectors**; LangChain also keeps each chunk’s full text in an **in-memory docstore** (so retrieval returns `Document.page_content`, not only an id).

Run the cell below to assert every chunk string is present in the store.

In [3]:
from qbrain_rag.infrastructure.vector_store import (
    chunk_texts_materialized_in_store,
    indexed_document_count,
)

assert indexed_document_count(store) == len(chunks)
assert chunk_texts_materialized_in_store(store, chunks)
print("OK: docstore contains all", len(chunks), "chunk texts (matches FAISS vector count).")

OK: docstore contains all 30 chunk texts (matches FAISS vector count).
